# Un assistant qui répond à partir des documents de l'entreprise (RAG)

Un LLM ne connaît pas les documents internes d'une entreprise : interrogé sur eux, il invente une réponse plausible. La **génération augmentée par récupération** (RAG) lui fournit, avec la question, les extraits pertinents d'une base documentaire.

Ingrédients de cette démonstration, tous ouverts et exécutables sur Colab :

- un petit LLM « assistant », [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B)
- un modèle d'*embeddings* multilingue, [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base), qui transforme un texte en vecteur pour chercher par le sens

Un GPU est recommandé (`Exécution > Modifier le type d'exécution`). Sans GPU, le notebook charge à la place la version 0,6B du modèle : plus légère, mais ses réponses sont beaucoup moins fiables.

In [1]:
import warnings

import huggingface_hub
import torch
import transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore", message=r"(?s).*HF_TOKEN.*")
huggingface_hub.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

device = "cuda" if torch.cuda.is_available() else "cpu"
TAILLE_MODELE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B : assez léger pour un CPU

if device == "cpu":
  warnings.warn(
    "Aucun GPU détecté : le notebook charge le modèle 0,6B, dont les réponses sont de moins bonne qualité. "
    "Pour le modèle 1,7B, choisissez un GPU (Exécution > Modifier le type d'exécution), puis réexécutez le notebook."
  )

LLM = f"Qwen/Qwen3-{TAILLE_MODELE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32
MODELE_EMBEDDING = "intfloat/multilingual-e5-base"

tokenizer = AutoTokenizer.from_pretrained(LLM)
modele = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)
modele_embedding = SentenceTransformer(MODELE_EMBEDDING, device=device)
print(device, TAILLE_MODELE)

cuda 1.7B


## La base documentaire

Le guide interne (fictif) de l'entreprise Acme : chaque paragraphe devient un extrait (*chunk*), préfixé par le titre de sa section.

In [2]:
DOCUMENT = """
Télétravail
Les salariés en CDI peuvent télétravailler jusqu'à 2 jours par semaine, après accord de leur responsable.
La demande se fait dans l'outil RH « Kiosque », au moins 15 jours avant le premier jour de télétravail.
Acme fournit un écran et un siège ergonomique aux salariés qui télétravaillent au moins 1 jour par semaine.

Congés
Chaque salarié dispose de 25 jours de congés payés et de 11 jours de RTT par an.
Les congés de plus de 5 jours consécutifs se posent dans Kiosque au moins 1 mois à l'avance.

Notes de frais
Le repas lors d'un déplacement professionnel est remboursé dans la limite de 23,50 € par repas, sur justificatif.
Les notes de frais doivent être déposées dans Kiosque dans les 45 jours suivant la dépense.
Le remboursement intervient sur le salaire du mois suivant la validation.

Déplacements
Le train est obligatoire pour les trajets de moins de 4 h, en seconde classe.
Pour les trajets de plus de 4 h, l'avion en classe économique est autorisé, sur accord du directeur de département.
La nuit d'hôtel est remboursée jusqu'à 140 € à Paris et 95 € en province.

Sécurité informatique
L'authentification à deux facteurs est obligatoire sur tous les outils de l'entreprise.
Il est interdit de copier des données clients dans un outil d'IA public ; utiliser l'assistant interne « AcmeGPT ».
Tout courriel suspect doit être transféré à l'adresse securite@acme.example.

Formation
Chaque salarié a droit à 3 jours de formation par an, en plus de son CPF.
Les demandes de formation sont validées lors de l'entretien annuel.
"""

In [3]:
passages = []
for section in DOCUMENT.strip().split("\n\n"):
  titre, *paragraphes = section.strip().splitlines()
  passages.extend(f"{titre} — {paragraphe}" for paragraphe in paragraphes)
print(len(passages))
passages[:3]

16


["Télétravail — Les salariés en CDI peuvent télétravailler jusqu'à 2 jours par semaine, après accord de leur responsable.",
 "Télétravail — La demande se fait dans l'outil RH « Kiosque », au moins 15 jours avant le premier jour de télétravail.",
 'Télétravail — Acme fournit un écran et un siège ergonomique aux salariés qui télétravaillent au moins 1 jour par semaine.']

## Sans RAG : le modèle invente

Branchons le modèle comme assistant RH, sans lui donner le guide, qu'il n'a jamais vu :

In [4]:
ROLE = (
  "Tu es l'assistant RH d'Acme. "
  "Réponds aux questions des salariés de façon précise et concise."
)
QUESTION = "Quel est le plafond de remboursement d'un repas lors d'un déplacement ?"


def interroge(messages: list[dict]) -> str:
  entrees = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
    return_tensors="pt",
  ).to(device)
  with torch.no_grad():
    sortie = modele.generate(
      **entrees,
      max_new_tokens=400,
      do_sample=False,
      temperature=None,
      top_p=None,
      top_k=None,
    )
  longueur_prompt = entrees["input_ids"].shape[1]
  return tokenizer.decode(sortie[0, longueur_prompt:], skip_special_tokens=True)


print(
  interroge(
    [{"role": "system", "content": ROLE}, {"role": "user", "content": QUESTION}]
  )
)

Le plafond de remboursement d'un repas lors d'un déplacement varie selon les réglementations de l'entreprise et les conventions de l'entreprise. En général, il est fixé à **15 € par repas** (un repas = un repas complet, comme un diner ou un déjeuner). Cela inclut les repas en établissement ou en déplacement. Veuillez consulter votre contrat de travail ou votre règlement interne pour des informations précises.


Le modèle ne peut pas connaître la réponse : s'il donne un montant, il l'invente (le guide dit 23,50 €). C'est une **hallucination**. Il avoue parfois ne pas savoir, mais on ne peut pas compter dessus.

## Rechercher les extraits pertinents

On calcule le vecteur de chaque extrait une fois pour toutes. Pour une question, on garde les extraits dont le vecteur est le plus proche du sien (score de similarité entre -1 et 1 : plus il est élevé, plus le sens est proche) :

In [5]:
# multilingual-e5 attend un préfixe : « passage: » devant un extrait,
# « query: » devant une question
embeddings_passages = modele_embedding.encode(
  ["passage: " + passage for passage in passages], normalize_embeddings=True
)


def cherche_passages(question: str, k: int = 3) -> list[tuple[float, str]]:
  embedding_question = modele_embedding.encode(
    "query: " + question, normalize_embeddings=True
  )
  scores = embeddings_passages @ embedding_question
  meilleurs = scores.argsort()[::-1][:k]
  return [(float(scores[i]), passages[i]) for i in meilleurs]


for score, passage in cherche_passages(QUESTION):
  print(f"{score:.2f}  {passage}")

0.89  Notes de frais — Le repas lors d'un déplacement professionnel est remboursé dans la limite de 23,50 € par repas, sur justificatif.
0.84  Déplacements — La nuit d'hôtel est remboursée jusqu'à 140 € à Paris et 95 € en province.
0.82  Notes de frais — Le remboursement intervient sur le salaire du mois suivant la validation.


## Avec RAG : répondre à partir des extraits

On donne au modèle les extraits trouvés, avec la consigne de s'y tenir et de citer ses sources. La fonction `rag` affiche aussi les extraits retrouvés et leur score :

In [6]:
INSTRUCTIONS = (
  "Réponds uniquement à partir des extraits fournis, "
  "en citant le numéro de l'extrait utilisé. Si la réponse n'y figure pas, "
  "dis que tu ne sais pas."
)
LIBELLE_EXTRAITS = "Extraits :"
LIBELLE_QUESTION = "Question :"


def rag(question: str) -> str:
  resultats = cherche_passages(question)
  print("Extraits retrouvés (score, texte) :")
  for i, (score, passage) in enumerate(resultats):
    print(f"[{i + 1}] {score:.2f}  {passage}")
  print("\nRéponse :")
  extraits = "\n".join(
    f"[{i + 1}] {passage}" for i, (_, passage) in enumerate(resultats)
  )
  return interroge(
    [
      {"role": "system", "content": f"{ROLE} {INSTRUCTIONS}"},
      {
        "role": "user",
        "content": f"{LIBELLE_EXTRAITS}\n{extraits}\n\n{LIBELLE_QUESTION} {question}",
      },
    ]
  )


print(rag(QUESTION))

Extraits retrouvés (score, texte) :
[1] 0.89  Notes de frais — Le repas lors d'un déplacement professionnel est remboursé dans la limite de 23,50 € par repas, sur justificatif.
[2] 0.84  Déplacements — La nuit d'hôtel est remboursée jusqu'à 140 € à Paris et 95 € en province.
[3] 0.82  Notes de frais — Le remboursement intervient sur le salaire du mois suivant la validation.

Réponse :
Le plafond de remboursement d'un repas lors d'un déplacement est de 23,50 €. Cela est indiqué dans l'extrait [1].


## Une question hors de la base

Le guide ne dit rien des voitures de fonction. Un bon assistant doit le reconnaître plutôt qu'inventer :

In [7]:
QUESTION_HORS_BASE = "Quelle est la politique d'Acme sur les voitures de fonction ?"

print(rag(QUESTION_HORS_BASE))

Extraits retrouvés (score, texte) :
[1] 0.81  Télétravail — Acme fournit un écran et un siège ergonomique aux salariés qui télétravaillent au moins 1 jour par semaine.
[2] 0.81  Sécurité informatique — Il est interdit de copier des données clients dans un outil d'IA public ; utiliser l'assistant interne « AcmeGPT ».
[3] 0.80  Sécurité informatique — Tout courriel suspect doit être transféré à l'adresse securite@acme.example.

Réponse :
Je ne sais pas quelle est la politique d'Acme sur les voitures de fonction, car cela n'est pas mentionné dans les extraits fournis.


Regardez les extraits affichés : la recherche renvoie toujours les `k` plus proches, même s'ils ne parlent pas de la question. Il n'y a pas de seuil de score. C'est donc la consigne « si la réponse n'y figure pas, dis que tu ne sais pas » qui doit empêcher le modèle d'inventer ; écarter les extraits sous un score minimal serait une protection de plus (mais avec ce modèle, les scores restent tous élevés, autour de 0,8 : le seuil se règle sur des exemples). Un petit modèle ne suit pas toujours la consigne : jugez la réponse affichée ci-dessus.

## Une question formulée autrement

La recherche se fait par le sens, pas par les mots exacts : « déjeuner en mission » retrouve l'extrait sur le repas en déplacement.

In [8]:
QUESTION_AUTRE = "Combien puis-je dépenser pour déjeuner quand je suis en mission ?"

print(rag(QUESTION_AUTRE))

Extraits retrouvés (score, texte) :
[1] 0.86  Notes de frais — Le repas lors d'un déplacement professionnel est remboursé dans la limite de 23,50 € par repas, sur justificatif.
[2] 0.82  Déplacements — La nuit d'hôtel est remboursée jusqu'à 140 € à Paris et 95 € en province.
[3] 0.81  Notes de frais — Les notes de frais doivent être déposées dans Kiosque dans les 45 jours suivant la dépense.

Réponse :
Le montant maximum autorisé pour un déjeuner lors d'un déplacement professionnel est de 23,50 € selon l'extrait [1].


## À retenir

- Sans accès aux documents, un LLM hallucine avec aplomb
- Le RAG combine une recherche par le sens et une génération guidée par les extraits trouvés
- Citer les sources permet de vérifier les réponses
- La qualité dépend d'abord de la base documentaire (à jour, bien découpée) et de la recherche : une mauvaise recherche donne une mauvaise réponse
- Même avec le RAG, un petit modèle peut se tromper : il faut évaluer sur ses propres questions avant de déployer